In [1]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. LOCAL PATH DEFINITIONS
DATASET_PATH = "./js_vulnllm_dataset.jsonl"
OUTPUT_DIR = "./vulnllm_js_checkpoints"
METRICS_FILE = "./training_metrics.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. LOAD TOKENIZER AND BASE MODEL
model_id = "Virtue-AI-HUB/VulnLLM-R-7B"
print(f"Loading Tokenizer & Base Model: {model_id} in 4-bit...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ Model successfully loaded into VRAM!")

Loading Tokenizer & Base Model: Virtue-AI-HUB/VulnLLM-R-7B in 4-bit...


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/home/u.js335405/.local/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model successfully loaded into VRAM!


In [6]:
import os
import csv
from datasets import load_dataset
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 1. CUSTOM METRICS LOGGER (For your Academic Paper)
class CSVLogCallback(TrainerCallback):
    """Custom callback to log metrics to a CSV file for LaTeX charting."""
    def __init__(self, log_path):
        self.log_path = log_path
        # Initialize CSV with headers
        with open(self.log_path, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Step', 'Epoch', 'Training_Loss', 'Validation_Loss', 'Learning_Rate'])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            with open(self.log_path, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    state.global_step,
                    round(state.epoch or 0, 2),
                    logs.get('loss', ''),
                    logs.get('eval_loss', ''),
                    logs.get('learning_rate', '')
                ])
            print(f"\n📊 Logged Metrics -> Step: {state.global_step} | Loss: {logs.get('loss', 'N/A')} | Eval Loss: {logs.get('eval_loss', 'N/A')}")

# 2. DATASET PREPARATION & SPLITTING (ChatML Aligned)
def format_prompts(example):
    is_vuln = example['target'] == 1
    
    # Constructing the message as a conversational array
    user_msg = f"Analyze the following {example['language']} code and identify if it contains a vulnerability. Return the reasoning and the prediction.\n\nCode:\n{example['code']}"
    
    if is_vuln:
        assistant_msg = f"<think>\n{example['reason']}\n</think>\n{example['human']}"
    else:
        assistant_msg = f"{example['human']}"
        
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg}
    ]
    
    # Apply Qwen2.5's native ChatML template using the tokenizer loaded in Cell 1
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    
    return {"text": formatted_text}

print(f"Loading Dataset from {DATASET_PATH}...")
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
dataset = dataset.map(format_prompts)

# Create 90/10 Train/Validation Split
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']
print(f"Data Split: {len(train_data)} Training Pairs | {len(val_data)} Validation Pairs")

# 3. LORA CONFIGURATION
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    lora_dropout=0.1, 
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] 
)
# REMOVED: model = get_peft_model(model, peft_config)
# The SFTTrainer handles the PEFT wrapping automatically when peft_config is passed.

# 4. TRAINING ARGUMENTS (Using SFTConfig)
training_args = SFTConfig(
    per_device_eval_batch_size=1,       # Reduce eval batch to absolute minimum
    eval_accumulation_steps=1,          # Move tensors to CPU faster during eval
    gradient_checkpointing=True,
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",          # Moved into SFTConfig per new API
    max_length=1500,                    # Renamed from max_seq_length per new API
    per_device_train_batch_size=2,      
    gradient_accumulation_steps=4,      
    optim="paged_adamw_32bit",
    eval_strategy="steps",              
    eval_steps=20,                      
    save_strategy="steps",              
    save_steps=20,                      
    logging_steps=10,                   
    learning_rate=2e-5,                 
    weight_decay=0.001,
    fp16=False,
    bf16=True,                          # <-- CHANGED: A30 Hardware Upgrade
    max_grad_norm=0.3,                  
    num_train_epochs=3,                 
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_total_limit=2,                 
    load_best_model_at_end=True,        
)

# 5. INITIALIZE SFT TRAINER
trainer = SFTTrainer(
    model=model,                        # Pass the raw base model from Cell 1
    train_dataset=train_data,
    eval_dataset=val_data,              
    peft_config=peft_config,            # The trainer uses this to wrap the model
    processing_class=tokenizer,         
    args=training_args,
    callbacks=[CSVLogCallback(METRICS_FILE)] 
)

# 6. EXECUTE TRAINING
print("\n🚀 Initiating LoRA Fine-Tuning...")
# Resume from checkpoint if one exists in the output directory
checkpoint = None
if len(os.listdir(OUTPUT_DIR)) > 0:
    print("Found existing checkpoints! Resuming training...")
    checkpoint = True

trainer.train(resume_from_checkpoint=checkpoint)

# 7. SAVE THE FINAL ADAPTER
# <-- CHANGED: Removed BASE_DIR reliance for local HPC environment
final_save_path = "./final_js_vulnllm_adapter"
trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"✅ Training Complete! Final adapter safely stored at {final_save_path}")

Loading Dataset from ./js_vulnllm_dataset.jsonl...


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Data Split: 1333 Training Pairs | 149 Validation Pairs


/home/u.js335405/.local/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/u.js335405/.local/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.



🚀 Initiating LoRA Fine-Tuning...


Step,Training Loss,Validation Loss
20,2.430780,2.824451
40,2.012229,2.595527
60,2.037478,2.341375
80,1.653043,2.099577
100,1.681355,1.877654
120,1.506429,1.695291
140,1.555485,1.556007
160,1.392837,1.490182
180,1.387938,1.450017
200,1.327939,1.418704



📊 Logged Metrics -> Step: 10 | Loss: 2.075999069213867 | Eval Loss: N/A

📊 Logged Metrics -> Step: 20 | Loss: 2.4307796478271486 | Eval Loss: N/A

📊 Logged Metrics -> Step: 20 | Loss: N/A | Eval Loss: 2.824450731277466

📊 Logged Metrics -> Step: 30 | Loss: 2.426040458679199 | Eval Loss: N/A

📊 Logged Metrics -> Step: 40 | Loss: 2.012228775024414 | Eval Loss: N/A

📊 Logged Metrics -> Step: 40 | Loss: N/A | Eval Loss: 2.595526933670044

📊 Logged Metrics -> Step: 50 | Loss: 1.9377069473266602 | Eval Loss: N/A

📊 Logged Metrics -> Step: 60 | Loss: 2.0374782562255858 | Eval Loss: N/A

📊 Logged Metrics -> Step: 60 | Loss: N/A | Eval Loss: 2.341374635696411

📊 Logged Metrics -> Step: 70 | Loss: 1.5816346168518067 | Eval Loss: N/A

📊 Logged Metrics -> Step: 80 | Loss: 1.6530433654785157 | Eval Loss: N/A

📊 Logged Metrics -> Step: 80 | Loss: N/A | Eval Loss: 2.099576950073242

📊 Logged Metrics -> Step: 90 | Loss: 1.6142789840698242 | Eval Loss: N/A

📊 Logged Metrics -> Step: 100 | Loss: 1.6813